In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier, Pool
import lightgbm as lgb
import xgboost as xgb
from scipy.optimize import minimize
from scipy.stats import rankdata
import warnings
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR') # Réduire le bruit de TensorFlow

# ====================================================================
# ULTIMATE STACKING : CODE COMPLET AVEC 4 MODÈLES DE BASE (MLP inclus)
# ====================================================================

print("="*80)
print("ULTIMATE STACKING - AVEC MLP (Réseau Neuronal Profond)")
print("="*80)

# =========================
# 1. CHARGEMENT ET PRÉPARATION DES DONNÉES
# =========================
print("\n Chargement et Préparation des données...")

# Tente de charger la version avec Quantile Transformation (qt)
try:
    X_train = pd.read_csv('X_train_ultra_qt.csv')
    X_test = pd.read_csv('X_test_ultra_qt.csv')
except:
    # Charge la version standard si 'qt' n'existe pas
    X_train = pd.read_csv('X_train_ultra.csv')
    X_test = pd.read_csv('X_test_ultra.csv')

y_train = pd.read_csv('y_train.csv')['label'].values
test_ids = pd.read_csv('test_ids.csv')['challenge_id'].values

# Intégration optionnelle de CamemBERT (équivalent à la feature LLM)
try:
    X_train_cam = pd.read_csv('train_camembert_features_150.csv')
    X_test_cam = pd.read_csv('test_camembert_features_150.csv')
    X_train = pd.concat([X_train, X_train_cam], axis=1)
    X_test = pd.concat([X_test, X_test_cam], axis=1)
    print(f"Features LLM (CamemBERT) intégrées.")
except:
    pass

# NORMALISATION POUR LE MLP
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

# =========================
# 2. CONFIGURATION DES MODÈLES DE BASE (y compris MLP pour simuler CNN)
# =========================
print("\n Configuration des 4 modèles de base...")

# Configuration des modèles GBM (inchangée)
base_models_config = {
    'lgb': {
        'params': {
            'objective': 'binary', 'metric': 'binary_logloss', 'boosting_type': 'gbdt',
            'num_leaves': 127, 'learning_rate': 0.01, 'feature_fraction': 0.7,
            'bagging_fraction': 0.7, 'bagging_freq': 5, 'max_depth': 15,
            'min_child_samples': 30, 'reg_alpha': 0.2, 'reg_lambda': 0.2,
            'verbose': -1, 'random_state': 42,
        },
        'num_rounds': 10000, 'early_stopping': 300
    },
    'cat': {
        'params': {
            'iterations': 5000, 'learning_rate': 0.01, 'depth': 10, 'l2_leaf_reg': 5,
            'subsample': 0.7, 'colsample_bylevel': 0.7, 'min_data_in_leaf': 30,
            'loss_function': 'Logloss', 'random_seed': 42, 'verbose': False,
            'early_stopping_rounds': 300,
        }
    },
    'xgb': {
        'params': {
            'objective': 'binary:logistic', 'eval_metric': 'logloss', 'max_depth': 12,
            'learning_rate': 0.01, 'subsample': 0.7, 'colsample_bytree': 0.7,
            'min_child_weight': 5, 'reg_alpha': 0.2, 'reg_lambda': 0.2,
            'random_state': 42, 'tree_method': 'hist',
        },
        'num_rounds': 10000, 'early_stopping': 300
    }
}

# Fonction pour créer le modèle MLP (simule le CNN sur données tabulaires)
def create_mlp_model(input_dim):
    model = Sequential([
        Dense(256, activation='relu', input_shape=(input_dim,)),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC', 'accuracy'])
    return model

# =========================
# 3. NIVEAU 1: CROSS-VALIDATION POUR OOF PREDICTIONS (4 MODÈLES)
# =========================
print("\n" + "="*80)
print("NIVEAU 1: ENTRAÎNEMENT DES 4 MODÈLES DE BASE AVEC CV (7 FOLDS)")
print("="*80)

n_splits = 7
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

model_names = ['lgb', 'cat', 'xgb', 'mlp']
oof_predictions = {name: np.zeros(len(X_train)) for name in model_names}
test_predictions = {name: np.zeros(len(X_test)) for name in model_names}
fold_scores = {name: [] for name in model_names}

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\n{'='*20} FOLD {fold + 1}/{n_splits} {'='*20}")
    
    # Préparer les données pour les GBM
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    
    # Préparer les données pour le MLP (scaled)
    X_tr_sc, X_val_sc = X_train_scaled.iloc[train_idx], X_train_scaled.iloc[val_idx]

    # --- LightGBM, CatBoost, XGBoost (code inchangé) ---
    # ... (Omission du code GBM pour la concision ici, mais il serait inclus) ...
    
    # ---------------- LightGBM ----------------
    lgb_train = lgb.Dataset(X_tr, y_tr)
    lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)
    model_lgb = lgb.train(
        base_models_config['lgb']['params'], lgb_train, num_boost_round=base_models_config['lgb']['num_rounds'], 
        valid_sets=[lgb_train, lgb_val], 
        callbacks=[lgb.early_stopping(stopping_rounds=base_models_config['lgb']['early_stopping']), lgb.log_evaluation(10000)]
    )
    oof_predictions['lgb'][val_idx] = model_lgb.predict(X_val, num_iteration=model_lgb.best_iteration)
    test_predictions['lgb'] += model_lgb.predict(X_test, num_iteration=model_lgb.best_iteration) / n_splits
    fold_scores['lgb'].append(accuracy_score(y_val, (oof_predictions['lgb'][val_idx] > 0.5).astype(int)))
    # print(f"LGB Acc: {fold_scores['lgb'][-1]:.4f}")
    
    # ---------------- CatBoost ----------------
    model_cat = CatBoostClassifier(**base_models_config['cat']['params'])
    model_cat.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True, plot=False)
    oof_predictions['cat'][val_idx] = model_cat.predict_proba(X_val)[:, 1]
    test_predictions['cat'] += model_cat.predict_proba(X_test)[:, 1] / n_splits
    fold_scores['cat'].append(accuracy_score(y_val, (oof_predictions['cat'][val_idx] > 0.5).astype(int)))
    # print(f"CAT Acc: {fold_scores['cat'][-1]:.4f}")

    # ---------------- XGBoost ----------------
    xgb_train = xgb.DMatrix(X_tr, label=y_tr)
    xgb_val = xgb.DMatrix(X_val, label=y_val)
    xgb_test = xgb.DMatrix(X_test)
    model_xgb = xgb.train(
        base_models_config['xgb']['params'], xgb_train, num_boost_round=base_models_config['xgb']['num_rounds'],
        evals=[(xgb_train, 'train'), (xgb_val, 'val')], early_stopping_rounds=base_models_config['xgb']['early_stopping'],
        verbose_eval=10000
    )
    oof_predictions['xgb'][val_idx] = model_xgb.predict(xgb_val, iteration_range=(0, model_xgb.best_iteration))
    test_predictions['xgb'] += model_xgb.predict(xgb_test, iteration_range=(0, model_xgb.best_iteration)) / n_splits
    fold_scores['xgb'].append(accuracy_score(y_val, (oof_predictions['xgb'][val_idx] > 0.5).astype(int)))
    # print(f"XGB Acc: {fold_scores['xgb'][-1]:.4f}")

    # ---------------- MLP  ----------------
    input_dim = X_train.shape[1]
    model_mlp = create_mlp_model(input_dim)
    
    # Entraînement sur les données Scaled
    es = EarlyStopping(monitor='val_auc', mode='max', patience=50, restore_best_weights=True)
    model_mlp.fit(X_tr_sc, y_tr, validation_data=(X_val_sc, y_val), epochs=500, batch_size=64, callbacks=[es], verbose=0)
    
    # Prédiction
    oof_predictions['mlp'][val_idx] = model_mlp.predict(X_val_sc).flatten()
    test_predictions['mlp'] += model_mlp.predict(X_test_scaled).flatten() / n_splits
    fold_scores['mlp'].append(accuracy_score(y_val, (oof_predictions['mlp'][val_idx] > 0.5).astype(int)))
    # print(f"MLP Acc: {fold_scores['mlp'][-1]:.4f}")

# =========================
# 4. ANALYSE DES MODÈLES DE BASE
# =========================
print("\n" + "="*80)
print("ANALYSE DES 4 MODÈLES DE BASE (OOF SCORES)")
print("="*80)

oof_matrix = np.column_stack([oof_predictions[m] for m in model_names])
test_matrix = np.column_stack([test_predictions[m] for m in model_names])

for model_name in model_names:
    oof = oof_predictions[model_name]
    acc = accuracy_score(y_train, (oof > 0.5).astype(int))
    auc = roc_auc_score(y_train, oof)
    print(f"{model_name.upper():4} | OOF Acc: {acc:.4f} | OOF AUC: {auc:.4f} | CV Mean: {np.mean(fold_scores[model_name]):.4f}")



ULTIMATE STACKING - AVEC MLP (Réseau Neuronal Profond)

📂 Chargement et Préparation des données...
✓ Features LLM (CamemBERT) intégrées.
✓ X_train: (154914, 340) | X_test: (103380, 340)

⚙️ Configuration des 4 modèles de base...

NIVEAU 1: ENTRAÎNEMENT DES 4 MODÈLES DE BASE AVEC CV (7 FOLDS)

==================== FOLD 1/7 ====================
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[4058]	training's binary_logloss: 0.113521	valid_1's binary_logloss: 0.344447
[0]	train-logloss:0.68526	val-logloss:0.68572
[1647]	train-logloss:0.09181	val-logloss:0.33917
3231/3231 [==============================] - 8s 2ms/step

==================== FOLD 2/7 ====================
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[3354]	training's binary_logloss: 0.137521	valid_1's binary_logloss: 0.345209
[0]	train-logloss:0.68526	val-logloss:0.68568
[1640]	train-logloss:0.09304	val-logloss:0.34015
3231/

In [ ]:
# =========================
# 5. NIVEAU 2: META-LEARNERS (8 MÉTHODES)
# =========================
print("\n" + "="*80)
print("NIVEAU 2: CALCUL DES 8 MÉTHODES D'ENSEMBLE")
print("="*80)

# NOTE: oof_matrix et test_matrix sont supposées être définies ici.
# Si vous avez sauvegardé/chargé les prédictions, elles doivent être chargées avant.

# Initialisation
ensemble_methods = {}
initial_aucs = [roc_auc_score(y_train, oof_predictions[m]) for m in model_names]

# --- 1. MOYENNE SIMPLE ---
oof_avg = oof_matrix.mean(axis=1)
test_avg = test_matrix.mean(axis=1)
ensemble_methods['simple_average'] = (oof_avg, test_avg, accuracy_score(y_train, (oof_avg > 0.5).astype(int)), roc_auc_score(y_train, oof_avg))

# --- 2. MOYENNE PONDÉRÉE PAR AUC ---
weights_auc = np.array(initial_aucs) / sum(initial_aucs)
oof_weighted_auc = (oof_matrix * weights_auc).sum(axis=1)
test_weighted_auc = (test_matrix * weights_auc).sum(axis=1)
ensemble_methods['weighted_auc'] = (oof_weighted_auc, test_weighted_auc, accuracy_score(y_train, (oof_weighted_auc > 0.5).astype(int)), roc_auc_score(y_train, oof_weighted_auc))

# --- 3. RANK AVERAGING ---
oof_ranks = np.column_stack([rankdata(oof_predictions[m]) for m in model_names])
test_ranks = np.column_stack([rankdata(test_predictions[m]) for m in model_names])
oof_rank_avg = (oof_ranks.mean(axis=1) - oof_ranks.min()) / (oof_ranks.max() - oof_ranks.min())
test_rank_avg = (test_ranks.mean(axis=1) - test_ranks.min()) / (test_ranks.max() - test_ranks.min())
ensemble_methods['rank_average'] = (oof_rank_avg, test_rank_avg, accuracy_score(y_train, (oof_rank_avg > 0.5).astype(int)), roc_auc_score(y_train, oof_rank_avg))

# --- 4. OPTIMISATION NUMÉRIQUE (SLSQP) ---
def objective_auc(weights):
    weights = weights / weights.sum()
    oof_blend = (oof_matrix * weights).sum(axis=1)
    return -roc_auc_score(y_train, oof_blend)
initial_weights = np.array([1/len(model_names)] * len(model_names))
bounds = [(0, 1)] * len(model_names)
constraints = {'type': 'eq', 'fun': lambda w: w.sum() - 1}
result = minimize(objective_auc, initial_weights, method='SLSQP', bounds=bounds, constraints=constraints)
optimal_weights_slsqp = result.x
oof_optimized_slsqp = (oof_matrix * optimal_weights_slsqp).sum(axis=1)
test_optimized_slsqp = (test_matrix * optimal_weights_slsqp).sum(axis=1)
ensemble_methods['optimized_slsqp'] = (oof_optimized_slsqp, test_optimized_slsqp, accuracy_score(y_train, (oof_optimized_slsqp > 0.5).astype(int)), roc_auc_score(y_train, oof_optimized_slsqp))

# --- 5. LOGISTIC REGRESSION META-LEARNER ---
lr_meta = LogisticRegression(C=0.1, random_state=42, max_iter=1000, solver='lbfgs')
lr_meta.fit(oof_matrix, y_train)
oof_lr = lr_meta.predict_proba(oof_matrix)[:, 1]
test_lr = lr_meta.predict_proba(test_matrix)[:, 1]
ensemble_methods['logistic_regression'] = (oof_lr, test_lr, accuracy_score(y_train, (oof_lr > 0.5).astype(int)), roc_auc_score(y_train, oof_lr))

# --- 6. MOYENNE GÉOMÉTRIQUE ---
meta_train_safe = np.clip(oof_matrix, 1e-10, 1 - 1e-10)
meta_test_safe = np.clip(test_matrix, 1e-10, 1 - 1e-10)
oof_geomean = np.prod(meta_train_safe, axis=1) ** (1/meta_train_safe.shape[1])
test_geomean = np.prod(meta_test_safe, axis=1) ** (1/meta_test_safe.shape[1])
ensemble_methods['geometric_mean'] = (oof_geomean, test_geomean, accuracy_score(y_train, (oof_geomean > 0.5).astype(int)), roc_auc_score(y_train, oof_geomean))

# --- 7. POWER MEAN (p=2) ---
p = 2
oof_powermean = ((oof_matrix ** p).mean(axis=1)) ** (1/p)
test_powermean = ((test_matrix ** p).mean(axis=1)) ** (1/p)
ensemble_methods['power_mean'] = (oof_powermean, test_powermean, accuracy_score(y_train, (oof_powermean > 0.5).astype(int)), roc_auc_score(y_train, oof_powermean))

# --- 8. RIDGE REGRESSION META-LEARNER (plus robuste) ---
from sklearn.linear_model import Ridge
ridge_meta = Ridge(alpha=1.0, random_state=42)
ridge_meta.fit(oof_matrix, y_train)
oof_ridge = ridge_meta.predict(oof_matrix)
oof_ridge = np.clip(oof_ridge, 0, 1) # Assurer que les probas restent dans [0, 1]
test_ridge = ridge_meta.predict(test_matrix)
test_ridge = np.clip(test_ridge, 0, 1)
ensemble_methods['ridge_regression'] = (oof_ridge, test_ridge, accuracy_score(y_train, (oof_ridge > 0.5).astype(int)), roc_auc_score(y_train, oof_ridge))


# =========================
# 6. COMPARAISON ET SÉLECTION
# =========================
methods_sorted = sorted(ensemble_methods.items(), key=lambda x: x[1][3], reverse=True)
best_name, (best_oof, best_test, best_acc, best_auc) = methods_sorted[0]

print("\n Classement des ensembles par AUC OOF:")
for i, (name, (_, _, acc, auc)) in enumerate(methods_sorted, 1):
    print(f"  {i}. {name.upper():25} → Acc: {acc:.4f} | AUC: {auc:.4f}")


# =========================
# 7 & 8. OPTIMISATION DU SEUIL ET GÉNÉRATION DES FICHIERS
# =========================
print("\n" + "="*80)
print("OPTIMISATION DU SEUIL & GÉNÉRATION DES FICHIERS DE SOUMISSION")
print("1. 8 fichiers pour chaque MÉTHODE D'ENSEMBLE (avec son seuil optimal)")
print("2. 4 fichiers pour chaque ALGO INDIVIDUEL (avec le seuil de l'ensemble optimal)")
print("="*80)

# Définition de la plage de seuils
thresholds = np.arange(0.35, 0.65, 0.001)
ensemble_results = {}
best_global_threshold = 0.5 

# --- A. Soumissions pour les 8 Méthodes d'Ensemble (Seuil Spécifique à Chacune) ---
print("\n GÉNÉRATION DES ENSEMBLES (Seuil Spécifique):")

for name, (oof_proba, test_proba, acc_base, auc) in ensemble_methods.items():
    
    # 1. Optimisation du seuil spécifique à la méthode
    best_thresh_for_method = 0.5
    best_acc_for_method = 0
    
    for thresh in thresholds:
        current_acc = accuracy_score(y_train, (oof_proba > thresh).astype(int))
        if current_acc > best_acc_for_method:
            best_acc_for_method = current_acc
            best_thresh_for_method = thresh
            
    ensemble_results[name] = {
        'threshold': best_thresh_for_method,
        'accuracy': best_acc_for_method,
        'auc': auc,
        'test_proba': test_proba
    }

    # 2. Génération du fichier de soumission binaire (Format ID, Prediction)
    final_predictions_binary = (test_proba > best_thresh_for_method).astype(int)
    
    submission_df = pd.DataFrame({
        'ID': test_ids,
        'Prediction': final_predictions_binary
    })
    
    # Nom du fichier : submission_ensemble_[nom_methode].csv
    filename = f'submission_ensemble_{name.lower()}.csv'
    submission_df.to_csv(filename, index=False)
    
    print(f"    {name.upper():20} | Acc OOF: {best_acc_for_method:.4f} (T={best_thresh_for_method:.4f}) -> {filename}")

# Détermination du meilleur ensemble pour le nommage ULTIMATE
best_ensemble_name = max(ensemble_results, key=lambda k: ensemble_results[k]['auc'])
best_global_threshold = ensemble_results[best_ensemble_name]['threshold'] # Seuil du modèle avec le meilleur AUC

# 3. Création du fichier ULTIMATE (le meilleur des 8)
best_ensemble_test_proba = ensemble_results[best_ensemble_name]['test_proba']
final_predictions_ultimate = (best_ensemble_test_proba > best_global_threshold).astype(int)

submission_ultimate_ensemble = pd.DataFrame({
    'ID': test_ids,
    'Prediction': final_predictions_ultimate
})
submission_ultimate_ensemble.to_csv('submission_ultimate_ensemble.csv', index=False)
print(f"\n FICHIER ULTIMATE (basé sur {best_ensemble_name.upper()}) | Seuil: {best_global_threshold:.4f} -> submission_ultimate_ensemble.csv")


# --- B. Soumissions pour les Modèles de Base Individuels (avec le meilleur seuil global) ---
print(f"\n GÉNÉRATION DES ALGOS INDIVIDUELS (avec T={best_global_threshold:.4f}):")

# Mappage pour assurer la convention de nommage 'nn' pour 'mlp'
individual_models_map = {
    'lgb': test_predictions['lgb'],
    'cat': test_predictions['cat'],
    'xgb': test_predictions['xgb'],
    'mlp': test_predictions['mlp'], 
}

for name, test_proba in individual_models_map.items():
    
    # Prédire binarisé en utilisant le meilleur seuil global (celui de l'ensemble optimal)
    final_predictions_binary = (test_proba > best_global_threshold).astype(int)
    
    # Remplacer 'mlp' par 'nn' pour le nommage du fichier
    filename_name = 'nn' if name == 'mlp' else name
    
    submission_individual = pd.DataFrame({
        'ID': test_ids,
        'Prediction': final_predictions_binary
    })
    
    # Nom du fichier : submission_ultimate_lgb.csv, submission_ultimate_cat.csv, etc.
    filename = f'submission_ultimate_{filename_name.lower()}.csv'
    submission_individual.to_csv(filename, index=False)
    print(f"   {filename_name.upper():20} -> {filename}")

print("\n Stacking Ultime Terminé. 13 fichiers de soumission créés (8 ensembles + 4 individuels + 1 ultimate)!")


NIVEAU 2: CALCUL DES 8 MÉTHODES D'ENSEMBLE

🏆 Classement des ensembles par AUC OOF:
  1. RIDGE_REGRESSION          → Acc: 0.8521 | AUC: 0.9251
  2. LOGISTIC_REGRESSION       → Acc: 0.8513 | AUC: 0.9248
  3. OPTIMIZED_SLSQP           → Acc: 0.8501 | AUC: 0.9235
  4. GEOMETRIC_MEAN            → Acc: 0.8486 | AUC: 0.9223
  5. WEIGHTED_AUC              → Acc: 0.8486 | AUC: 0.9222
  6. SIMPLE_AVERAGE            → Acc: 0.8486 | AUC: 0.9222
  7. RANK_AVERAGE              → Acc: 0.8415 | AUC: 0.9222
  8. POWER_MEAN                → Acc: 0.8484 | AUC: 0.9219

OPTIMISATION DU SEUIL & GÉNÉRATION DES FICHIERS DE SOUMISSION
1. 8 fichiers pour chaque MÉTHODE D'ENSEMBLE (avec son seuil optimal)
2. 4 fichiers pour chaque ALGO INDIVIDUEL (avec le seuil de l'ensemble optimal)

📝 GÉNÉRATION DES ENSEMBLES (Seuil Spécifique):
  ✅ SIMPLE_AVERAGE       | Acc OOF: 0.8487 (T=0.5080) -> submission_ensemble_simple_average.csv
  ✅ WEIGHTED_AUC         | Acc OOF: 0.8487 (T=0.5070) -> submission_ensemble_weighted_